加入multi-query: 由于用户输入的原始问题往往存在模糊性或关键词缺失，通过 LLM 等手段 将一个问题改写成多个视角不同的等价问题，再进行并发检索，最后合并结果


加入rerank: 搜索系统有两部分 一是召回 通过模型从数据库大量的数据中找到相近的10~50条内容, 计算代价小,但是由于这个召回的过程是embedding的匹配的过程,所以对于细节部分可能处理不好,是粗粒度的搜索; 二是重排(rerank), 采用交叉编码器架构, 全交互注意力, 计算代价大, 负责将召回的数据进行排序,将匹配度高的返回给用户.


为什么计算代价有区别: 向量匹配只需要算两个向量的乘积得出相似度就可以了, 因为是单纯的向量相乘所以没有语义能力, 重排是计算注意力,计算量大

In [4]:
import torch
print(torch.__version__)

2.8.0+cu128


In [4]:
import os
import json
import torch
import faiss
import pickle
import librosa
import numpy as np
from tqdm import tqdm
from pathlib import Path
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info

# ==========================================
# 1. Faiss Index
# ==========================================
class FaissIndex:
    def __init__(self, dim):
        self.index = faiss.IndexFlatIP(dim)
        self.metadata = []

    def add(self, vectors, metas):
        vectors = np.ascontiguousarray(vectors).astype("float32")
        self.index.add(vectors)
        self.metadata.extend(metas)

    def save(self, path):
        faiss.write_index(self.index, f"{path}.index")
        with open(f"{path}.meta", "wb") as f:
            pickle.dump(self.metadata, f)

# ==========================================
# 2. Qwen2.5 Omni Audio Caption（修正版）
# ==========================================
class QwenAudioCaptioner:
    def __init__(self, model_path, device="cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        
        # 使用官方支持的类加载模型和处理器
        self.model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
            model_path,
            device_map="auto",
            torch_dtype=torch.float16,
            trust_remote_code=True,
            
        ).eval()
        
        self.processor = Qwen2_5OmniProcessor.from_pretrained(
            model_path,
            trust_remote_code=True
        )
        
        # 关键优化：禁用语音生成以节省约2GB显存[reference:2]
        # 因为我们只需要文本输出
        self.model.disable_talker()

    def caption(self, audio_path):
        # 按照官方示例构建对话结构
        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "audio", "audio": audio_path},
                    {"type": "text", "text": "Audio transcription and event description:"}
                ]
            }
        ]
        
        # 应用聊天模板
        prompt = self.processor.apply_chat_template(
            conversation, 
            add_generation_prompt=True, 
            tokenize=False
        )
        
        # 使用官方工具函数处理多模态信息
        audios, images, videos = process_mm_info(conversation, use_audio_in_video=False)
        
        # 处理输入，注意参数名为单数 'audio' 而非 'audios'[reference:3]
        inputs = self.processor(
            text=prompt, 
            audio=audios, 
            return_tensors="pt", 
            padding=True
        ).to(self.model.device)
        
        # 推理生成，因为调用了 disable_talker()，必须设置 return_audio=False[reference:4]
        with torch.no_grad():
            generated_ids = self.model.generate(
                **inputs,
                return_audio=False,  # 强制返回文本
                max_new_tokens=64,
                eos_token_id=self.processor.tokenizer.eos_token_id,  # 避免生成问题
                pad_token_id=self.processor.tokenizer.pad_token_id
            )
        
        # 解码生成的文本
        text = self.processor.batch_decode(
            generated_ids, 
            skip_special_tokens=True, 
            clean_up_tokenization_spaces=False
        )[0]
        
        # 后处理：清理输出，只保留模型回答部分
        if "assistant" in text:
            text = text.split("assistant")[-1].strip()
        
        return text

# ==========================================
# 3. Caption 缓存
# ==========================================
CACHE_PATH = "caption_cache.json"

if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, "r") as f:
        caption_cache = json.load(f)
else:
    caption_cache = {}

def safe_caption(captioner, path):
    if path in caption_cache:
        return caption_cache[path]

    try:
        text = captioner.caption(path)

        if len(text) < 3:
            raise ValueError("Generated caption too short")

        caption_cache[path] = text
        with open(CACHE_PATH, "w") as f:
            json.dump(caption_cache, f, ensure_ascii=False, indent=2)

        #print(f"[CAPTION] {text}")
        return text

    except Exception as e:
        print(f"[ERROR] Failed for {path}: {e}")
        name = os.path.basename(path)
        fallback = name.replace("_", " ").replace(".wav", "")
        caption_cache[path] = fallback
        return fallback

# ==========================================
# 4. CLAP Wrapper（保持不变，已验证可用）
# ==========================================
from transformers import ClapProcessor, ClapModel

class CLAPWrapper:
    def __init__(self, model_path, device="cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        self.model = ClapModel.from_pretrained(model_path).to(self.device)
        self.processor = ClapProcessor.from_pretrained(model_path)

    def get_audio_embedding(self, audio):
        inputs = self.processor(
            audio=[audio],
            return_tensors="pt",
            sampling_rate=48000
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model.get_audio_features(**inputs)
            emb = outputs.pooler_output.cpu().numpy()

        return emb

# ==========================================
# 5. 音频加载（保持不变，已验证可用）
# ==========================================
def load_audio(path, sr=48000, max_duration=10):
    y, _ = librosa.load(path, sr=sr, mono=True)

    max_len = sr * max_duration
    if len(y) > max_len:
        y = y[:max_len]
    elif len(y) < max_len:
        y = np.pad(y, (0, max_len - len(y)))

    return y

# ==========================================
# 6. 构建流程
# ==========================================
AUDIO_DIR = "data/raw_audio"
CLAP_MODEL_PATH = "/root/autodl-tmp/clap-model"  # 替换为你的CLAP模型路径
QWEN_MODEL_PATH = "/root/autodl-tmp/qwen2.5-omni-3b"
SAVE_PATH = "data/embeddings/audio_index"
print("Loading CLAP model...")
clap_model = CLAPWrapper(CLAP_MODEL_PATH)
    
print("Loading Qwen Audio Captioner...")
captioner = QwenAudioCaptioner(QWEN_MODEL_PATH)
def build():
    

    vectors = []
    metas = []

    files = list(Path(AUDIO_DIR).rglob("*.wav"))
    print(f"Found {len(files)} audio files")

    for path_obj in tqdm(files):
        path_str = str(path_obj)

        try:
            # Step 1: Generate CLAP embedding
            audio = load_audio(path_str)
            emb = clap_model.get_audio_embedding(audio)
            vectors.append(emb[0].astype("float32"))

            # Step 2: Generate text description with Qwen
            desc = safe_caption(captioner, path_str)

            metas.append({
                "path": path_str,
                "description": desc
            })

        except Exception as e:
            print(f"[SKIP] {path_str}: {e}")

    vectors_np = np.stack(vectors)
    faiss.normalize_L2(vectors_np)

    index = FaissIndex(dim=vectors_np.shape[1])
    index.add(vectors_np, metas)
    index.save(SAVE_PATH)

    print("✅ Index building completed successfully")



Loading CLAP model...


Loading weights:   0%|          | 0/447 [00:00<?, ?it/s]

Loading Qwen Audio Captioner...


Loading weights:   0%|          | 0/2543 [00:00<?, ?it/s]

Qwen2_5OmniForConditionalGeneration LOAD REPORT from: /root/autodl-tmp/qwen2.5-omni-3b
Key                                                | Status     |  | 
---------------------------------------------------+------------+--+-
token2wav.code2wav_dit_model.rotary_embed.inv_freq | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
import logging
# 禁用 root logger 的警告输出
logging.getLogger().setLevel(logging.ERROR)
if __name__ == "__main__":
    build()

Found 2000 audio files


  0%|          | 0/2000 [00:00<?, ?it/s]

 16%|█▋        | 327/2000 [00:38<03:16,  8.53it/s]


KeyboardInterrupt: 

In [ ]:
import faiss
import numpy as np
from collections import defaultdict



INDEX_PATH = "data/embeddings/audio_index"
MODEL_PATH = "/root/autodl-tmp/clap-model"


# ==========================================
# 1. Query扩展（Multi-Query）
# ==========================================
def expand_query(query):
    query = query.lower()
    expansions = [query]

    rules = {
        "ocean": ["sea", "wave", "beach"],
        "sea": ["ocean", "wave"],
        "rain": ["rainfall", "water drops"],
        "dog": ["barking", "puppy"],
        "fire": ["flame", "burning"],
        "wind": ["air", "storm"],
    }

    for k, vs in rules.items():
        if k in query:
            expansions.extend(vs)

    return list(set(expansions))


# ==========================================
# 2. 多查询检索
# ==========================================
def multi_search(query, model, index, topk=5):
    queries = expand_query(query)

    all_results = []

    for q in queries:
        emb = model.get_text_embedding(q)

        # 归一化（必须）
        faiss.normalize_L2(emb)

        D, I = index.index.search(emb.astype("float32"), topk)

        for idx, score in zip(I[0], D[0]):
            all_results.append((idx, float(score)))

    return all_results


# ==========================================
# 3. 结果融合（关键）
# ==========================================
def merge_results(results):
    score_dict = defaultdict(float)

    for idx, score in results:
        score_dict[idx] += score  # 累加

    ranked = sorted(score_dict.items(), key=lambda x: x[1], reverse=True)
    return ranked


# ==========================================
# 4. 简单 Rerank
# ==========================================
def rerank(ranked, metadata, query, topk=5):
    final = []

    query_words = query.lower().split()

    for idx, score in ranked[:50]:  # 先取前50再rerank
        meta = metadata[idx]
        path = meta["path"].lower()

        bonus = 0.0

        # 规则1：路径匹配关键词
        for w in query_words:
            if w in path:
                bonus += 0.05

        final.append({
            "path": meta["path"],
            "score": score + bonus
        })

    final = sorted(final, key=lambda x: x["score"], reverse=True)

    return final[:topk]


# ==========================================
# 5. 最终接口
# ==========================================
def search(query, topk=5):
    model = CLAPWrapper(MODEL_PATH)

    index = FaissIndex(dim=512)
    index.load(INDEX_PATH)

    # multi-query 检索
    raw_results = multi_search(query, model, index, topk=topk)

    # 融合
    merged = merge_results(raw_results)

    # rerank
    final = rerank(merged, index.metadata, query, topk=topk)

    return final


# ==========================================
# 6. CLI测试
# ==========================================
if __name__ == "__main__":
    while True:
        q = input("\nQuery: ")
        res = search(q)

        for r in res:
            print(f"{r['score']:.4f}  {r['path']}")

Loading weights:   0%|          | 0/447 [00:00<?, ?it/s]

0.4733  data/raw_audio/esc50/3-51909-A-42.wav
0.4730  data/raw_audio/esc50/3-51376-A-42.wav
0.4700  data/raw_audio/esc50/3-58772-A-42.wav
0.4652  data/raw_audio/esc50/1-53467-A-47.wav
0.4597  data/raw_audio/esc50/4-80761-A-42.wav


Loading weights:   0%|          | 0/447 [00:00<?, ?it/s]

0.2500  data/raw_audio/esc50/1-54918-A-14.wav
0.2476  data/raw_audio/esc50/4-187769-A-14.wav
0.2441  data/raw_audio/esc50/4-181362-A-13.wav
0.2417  data/raw_audio/esc50/5-215179-A-13.wav
0.2357  data/raw_audio/esc50/4-202749-A-13.wav


Loading weights:   0%|          | 0/447 [00:00<?, ?it/s]

0.2567  data/raw_audio/esc50/3-187710-A-11.wav
0.2542  data/raw_audio/esc50/4-195497-B-11.wav
0.2522  data/raw_audio/esc50/3-164592-A-15.wav
0.2429  data/raw_audio/esc50/1-9841-A-13.wav
0.2315  data/raw_audio/esc50/3-164595-A-15.wav


Loading weights:   0%|          | 0/447 [00:00<?, ?it/s]

0.4385  data/raw_audio/esc50/4-154443-A-24.wav
0.4122  data/raw_audio/esc50/1-29680-A-21.wav
0.4054  data/raw_audio/esc50/2-109505-A-21.wav
0.3854  data/raw_audio/esc50/1-59324-A-21.wav
0.3838  data/raw_audio/esc50/5-252248-A-34.wav


In [6]:
import os
from dotenv import load_dotenv

load_dotenv()
DEEPSEEK_API_KEY = os.getenv("deepseek_api_key")

In [13]:
import faiss
import numpy as np
import torch
import json
import os
from collections import defaultdict
from functools import lru_cache
from openai import OpenAI  # DeepSeek 兼容 OpenAI SDK
from sentence_transformers import CrossEncoder

# ==========================================
# 0. 配置与初始化 (建议单例化)
# ==========================================

MODEL_PATH = "/root/autodl-tmp/clap-model"
# 推荐使用轻量级精排模型，如 BGE-Reranker-Base
RERANK_MODEL_NAME = "/root/autodl-tmp/bge-reranker-base" 

client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")
# 本地加载 Reranker
rerank_model = CrossEncoder(RERANK_MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu")
#print(rerank_model.model)
# ==========================================
# 1. 语义扩展 (DeepSeek API + Cache)
# ==========================================
@lru_cache(maxsize=128)
def expand_query_llm(query):
    """
    使用 DeepSeek 将原始 Query 扩展为多个同义/近义词，增加召回覆盖率。
    lru_cache 解决 API 慢和重复调用费用问题。
    """
    prompt = f"Please provide 3-4 English synonyms or related audio descriptions for the following search term: '{query}'. Output only the terms separated by commas."
    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=50,
            temperature=0.3
        )
        content = response.choices[0].message.content
        expansions = [q.strip() for q in content.split(",") if q.strip()]
        return list(set([query] + expansions))
    except Exception as e:
        print(f"API Error: {e}, falling back to original query.")
        return [query]

# ==========================================
# 2. 结果融合 (RRF 算法)
# ==========================================
def rrf_merge(multi_results, k=60):
    """
    Reciprocal Rank Fusion: 核心在于排名而非原始分值。
    multi_results: List of List, 每一项是单次检索的 (idx, score) 列表
    """
    rrf_score = defaultdict(float)
    for results in multi_results:
        for rank, (idx, _) in enumerate(results):
            # rank 从 0 开始，公式：1 / (k + rank + 1)
            rrf_score[idx] += 1.0 / (k + rank + 1)
    
    # 按照 RRF 得分排序
    merged = sorted(rrf_score.items(), key=lambda x: x[1], reverse=True)
    return merged

# ==========================================
# 3. 深度重排 (Cross-Encoder)
# ==========================================
def deep_rerank(merged_candidates, index_metadata, query, topk=5):
    """
    使用 Cross-Encoder 对 Top-N 候选进行精排。
    """
    if not merged_candidates:
        return []

    # 取 RRF 前 20-30 个进行精排，兼顾速度与精度
    candidates = merged_candidates[:30]
    idx_list = [c[0] for c in candidates]
    
    # 准备 Cross-Encoder 输入: (Query, Passage/Metadata) 对
    # 假设你的 metadata 存了描述信息，如果没有，可以用 path 代替
    input_pairs = []
    for idx in idx_list:
        meta = index_metadata[idx]
        desc = meta.get("description", meta["path"]) # 优先用描述
        input_pairs.append([query, desc])

    # 模型打分 (推理)
    scores = rerank_model.predict(input_pairs)
    
    final_results = []
    for i, score in enumerate(scores):
        final_results.append({
            "path": index_metadata[idx_list[i]]["path"],
            "score": float(score),
            "idx": idx_list[i]
        })
    
    return sorted(final_results, key=lambda x: x["score"], reverse=True)[:topk]

# ==========================================
# 4. 完整检索链路
# ==========================================
def search_pro(query, model, index, topk=5):
    # 1. DeepSeek 扩展
    queries = expand_query_llm(query)
    
    # 2. 多路并发检索 (此处可用线程池加速)
    all_search_results = []
    for q in queries:
        emb = model.get_text_embedding(q)
        faiss.normalize_L2(emb)
        D, I = index.index.search(emb.astype("float32"), 50) # 召回多一点给精排
        all_search_results.append(list(zip(I[0], D[0])))
    
    # 3. RRF 融合
    merged = rrf_merge(all_search_results)
    
    # 4. Cross-Encoder 精排
    final = deep_rerank(merged, index.metadata, query, topk=topk)
    
    return final

test = rerank_model.predict([
    ["dog barking", "a dog is barking loudly"],
    ["dog barking", "car engine noise"]
])
print(test)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: /root/autodl-tmp/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[9.997824e-01 8.952460e-05]


In [11]:
if __name__ == "__main__":
    # 注意：确保此处 model 和 index 已经正确实例化
    # 加载基础检索模型 (假设你已有 CLAPWrapper 类)
    INDEX_PATH = "data/embeddings/audio_index"
    MODEL_PATH = "/root/autodl-tmp/clap-model"
    model = CLAPWrapper(MODEL_PATH) 

    #加载索引 (假设你已有 FaissIndex 类)
    index = FaissIndex(dim=512)
    index.load(INDEX_PATH)
    print("\n--- 系统就绪，开始检索测试 ---")
    try:
        while True:
            user_q = input("\n请输入搜索关键词 (q退出): ").strip()
            if user_q.lower() in ['q', 'exit']: break
            if not user_q: continue

            results = search_pro(user_q, model, index)

            print(f"\n为您找到以下结果:")
            for i, res in enumerate(results):
                print(f"{i+1}. [{res['score']:.4f}] {res['path']}")
    except KeyboardInterrupt:
        print("\n程序终止")

Loading weights:   0%|          | 0/447 [00:00<?, ?it/s]


--- 系统就绪，开始检索测试 ---

为您找到以下结果:
1. [0.0000] data/raw_audio/esc50/1-85123-A-31.wav
2. [0.0000] data/raw_audio/esc50/5-117773-A-16.wav
3. [0.0000] data/raw_audio/esc50/4-181362-A-13.wav
4. [0.0000] data/raw_audio/esc50/2-85139-A-13.wav
5. [0.0000] data/raw_audio/esc50/5-261464-A-23.wav
